# Task 1: Install and Authenticate Databricks CLI using OAuth U2M

# Databricks CLI: Core Concepts & Windows Reference Guide

---

## Part 1: Core Concepts & Assignment Overview

### What is Databricks CLI?
The Databricks Command Line Interface (CLI) allows you to automate and manage your Databricks workspace from a Windows terminal (PowerShell/CMD) without using the web UI.

### Key Difference: U2M vs M2M Authentication

| Feature | U2M (User-to-Machine) | M2M (Machine-to-Machine) |
| :--- | :--- | :--- |
| **Who uses it?** | Human Developers / Students | Automated Scripts / CI/CD Pipelines |
| **Login Method** | Web browser login (`databricks auth login`) | Client ID + Client Secret |
| **Identity** | Your personal user account | Databricks Service Principal |
| **Use Case** | Local testing, daily tasks, assignments | Scheduled jobs, GitHub Actions, Azure DevOps |

---

## Part 2: Task 1 - Basic Setup & U2M Login

### Step 1: Install CLI on Windows
Open **PowerShell** as Administrator and run:
- `winget install Databricks.DatabricksCLI`

### Step 2: Authenticate using U2M
Run this command in PowerShell to log in via browser:
- `databricks auth login --host https://<your-workspace-url> --profile DEFAULT
`
### Step 3: Verify Authentication
- `databricks auth profiles`
- `databricks fs ls dbfs:/`

---

# Task 2 & 3 : Initialize a Declarative Automation Bundle (DAB) Project

---

## 1. Concept Overview: What is a Declarative Automation Bundle?

Databricks Asset Bundles (DABs) treat Databricks resources—such as Jobs, Pipelines, and Notebooks—as software source code. Instead of manually creating jobs in the Web UI, you declare them using standard YAML configuration files (`databricks.yml`).

---

## 2. Step-by-Step Execution Guide

### Step 1: Initialize the Bundle Project
Run this command in your local terminal (PowerShell, Command Prompt, or Bash) from the folder where you want to create your project:

- `databricks bundle init`

**Interactive Prompts Setup:**
1. **Template Selection:** Select `default-python` (or press Enter for default).
2. **Unique Name for Project:** Enter `my_job_bundle`.

---

### Step 2: Define the `databricks.yml` File
Open the generated `databricks.yml` file (or create one by hand in your project folder) and ensure it has one **job resource** defined as shown below:

    bundle:
      name: my_job_bundle

    targets:
      dev:
        mode: development
        default: true
        workspace:
          host: https://<your-workspace-url>

    resources:
      jobs:
        my_sample_job:
          name: "Task 2 Automated Workflow Job"
          tasks:
            - task_key: run_sample_notebook
              notebook_task:
                notebook_path: ./src/notebook.py
              job_cluster_key: dev_cluster

          job_clusters:
            - job_cluster_key: dev_cluster
              new_cluster:
                spark_version: 13.3.x-scala2.12
                node_type_id: Standard_DS3_v2
                num_workers: 1

---

### Step 3: Create the Source Notebook File
Create a subfolder named `src` inside your project directory and add a file named `notebook.py` with the following content:

# Databricks notebook source
print("Hello from Databricks Asset Bundle Job Task!")

---

### Step 4: Validate, Deploy, and Run the Bundle

1. **Validate project syntax:**
   - `databricks bundle validate`

2. **Deploy the job resource to your workspace:**
   - `databricks bundle deploy -t dev`

3. **Trigger the job run using the CLI:**
   - `databricks bundle run -t dev my_sample_job`

4. **Clean up resources from workspace:**
   - `databricks bundle destroy -t dev`

# Task 5 : Configure M2M (Service Principal) Authentication & Deploy

---

## 1. Setup Environment Variables for Service Principal

To authenticate the Databricks CLI using M2M instead of personal U2M credentials, export the following environment variables in your terminal (PowerShell on Windows):

    $env:DATABRICKS_HOST="https://<your-workspace-url>.cloud.databricks.com"
    $env:DATABRICKS_CLIENT_ID="<your-service-principal-client-id>"
    $env:DATABRICKS_CLIENT_SECRET="<your-service-principal-client-secret>"

> **Note:** The CLI automatically checks these environment variables first. When present, it bypasses your personal U2M login.

---

## 2. Verify M2M Authentication

Run the following command in your terminal to confirm that the CLI recognizes the Service Principal credentials:

    databricks auth env

**Expected Output:**
The CLI will output the active host and client ID associated with the Service Principal, confirming that M2M mode is active.

---

## 3. Deploy the Bundle using M2M Credentials

Navigate to your bundle directory and execute the deployment command:

    # Validate bundle syntax under M2M authentication
    databricks bundle validate -t dev

    # Deploy the bundle using Service Principal identity
    databricks bundle deploy -t dev

---

## 4. Verify Active Profile Identity

To verify that your CLI is using M2M instead of your personal U2M profile, list the configured profiles:

    databricks auth profiles

# Task 6: GitHub Actions Workflow for PR Bundle Validation


---

## 1. Concept & CI/CD Strategy
To ensure code quality and prevent invalid bundle configurations from reaching the main branch, a **Continuous Integration (CI)** pipeline is configured. On every Pull Request (PR), GitHub Actions spins up an automated runner, installs the Databricks CLI, and runs `databricks bundle validate`. No resources are deployed during this check.

---

## 2. GitHub Actions Workflow Definition (.github/workflows/bundle_validate.yml)

---
    name: Validate Databricks Asset Bundle

    on:
      pull_request:
        branches:
          - main
          - dev

    jobs:
      validate-bundle:
        name: Validate DAB Syntax
        runs-on: ubuntu-latest

        steps:
          # Step 1: Checkout repository code
          - name: Checkout Code
            uses: actions/checkout@v4

          # Step 2: Setup Databricks CLI
          - name: Setup Databricks CLI
            uses: databricks/setup-cli@v0.220.0

          # Step 3: Validate Bundle Syntax using M2M Credentials
          - name: Run Bundle Validate
            env:
              DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
              DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
              DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
            run: |
              databricks bundle validate -t dev

### Databricks Asset Bundle (DAB) CI/CD Workflow Architecture

```text
Developer creates Pull Request (PR)
       │
       ▼
GitHub Actions Wakes Up (Ubuntu Server Spawned)
       │
       ▼
1. Checkout Repo Code (Download's Repo)
2. Install Databricks CLI v0.220.0
3. Inject GitHub Secrets:
   - DATABRICKS_HOST
   - DATABRICKS_CLIENT_ID
   - DATABRICKS_CLIENT_SECRET
       │
       ▼
Executes: databricks bundle validate -t dev
       │
       ├─── Valid Syntax? ──────► Shows Green Checkmark (✓) on PR
       └─── Syntax Error? ──────► Fails Build (✕) & Blocks PR